# Circularity Site Decomposition

Tracks `attn_out`, `mlp_out`, and `resid_post` circularity over training to test the hypothesis
that MLP drives circularity throughout training while attention operates in parallel until grokking.

Key metric: **MLP alignment** = `resid_post_circularity − attn_out_circularity`
- Negative: MLP output **suppresses** circular structure (adds non-aligned circular patterns)
- Positive: MLP output **amplifies** circular structure

Grokking is hypothesized to be the moment MLP alignment flips from negative to positive.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from miscope.analysis.artifact_loader import ArtifactLoader

RESULTS_BASE = '../results/modulo_addition_1layer'

# Health spectrum: fast healthy → canon healthy → late → degraded
VARIANTS = [
    {
        'label': 'p109/s485/ds598',
        'dir': 'modulo_addition_1layer_p109_seed485_dseed598',
        'grokking_epoch': 5243,
        'health': 'healthy',
        'color': '#1f77b4',
    },
    {
        'label': 'p113/s999/ds598',
        'dir': 'modulo_addition_1layer_p113_seed999_dseed598',
        'grokking_epoch': 12448,
        'health': 'healthy',
        'color': '#2ca02c',
    },
    {
        'label': 'p101/s485/ds598',
        'dir': 'modulo_addition_1layer_p101_seed485_dseed598',
        'grokking_epoch': 16656,
        'health': 'late_grokker',
        'color': '#ff7f0e',
    },
    {
        'label': 'p101/s999/ds598',
        'dir': 'modulo_addition_1layer_p101_seed999_dseed598',
        'grokking_epoch': -1,
        'health': 'late_grokker',
        'color': '#d62728',
    },
    {
        'label': 'p59/s485/ds598',
        'dir': 'modulo_addition_1layer_p59_seed485_dseed598',
        'grokking_epoch': -1,
        'health': 'degraded',
        'color': '#9467bd',
    },
]

def load_circularity(variant_dir):
    """Load circularity timeseries for all sites from repr_geometry summary."""
    loader = ArtifactLoader(f'{RESULTS_BASE}/{variant_dir}/artifacts')
    summary = loader.load_summary('repr_geometry')
    return {
        'epochs': summary['epochs'],
        'resid_pre': summary['resid_pre_circularity'],
        'attn_out': summary['attn_out_circularity'],
        'mlp_out': summary['mlp_out_circularity'],
        'resid_post': summary['resid_post_circularity'],
    }

# Load all variants
circularity_data = {}
for v in VARIANTS:
    circularity_data[v['label']] = load_circularity(v['dir'])
    print(f"Loaded {v['label']}: {len(circularity_data[v['label']]['epochs'])} epochs")

## Cell 1: Full Circularity Timeseries — All Sites Per Variant

Each row shows one variant. Lines: `attn_out` (mid-stream residual, green), `mlp_out` (MLP delta output, red),
`resid_post` (final residual, purple). `resid_pre` is zero throughout (embedding provides no circularity).

Vertical dashed line marks grokking epoch where applicable.

In [ ]:
SITE_COLORS = {
    'attn_out': '#2ca02c',
    'mlp_out': '#d62728',
    'resid_post': '#9467bd',
}
SITE_LABELS = {
    'attn_out': 'Attn (mid-stream)',
    'mlp_out': 'MLP (output delta)',
    'resid_post': 'Resid Post',
}

n_variants = len(VARIANTS)
fig = make_subplots(
    rows=n_variants, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=[f"{v['label']} ({v['health']}, grok={v['grokking_epoch']})" for v in VARIANTS],
)

for row_idx, v in enumerate(VARIANTS, 1):
    data = circularity_data[v['label']]
    epochs = data['epochs']

    for site in ['attn_out', 'mlp_out', 'resid_post']:
        fig.add_trace(
            go.Scatter(
                x=epochs, y=data[site],
                mode='lines',
                name=SITE_LABELS[site],
                legendgroup=site,
                showlegend=(row_idx == 1),
                line=dict(color=SITE_COLORS[site], width=2),
                hovertemplate=f"{SITE_LABELS[site]}<br>Epoch %{{x}}<br>Circ: %{{y:.3f}}<extra></extra>",
            ),
            row=row_idx, col=1,
        )

    # Grokking marker
    if v['grokking_epoch'] > 0:
        fig.add_vline(
            x=v['grokking_epoch'],
            line=dict(color='black', dash='dash', width=1),
            row=row_idx, col=1,
        )

    fig.update_yaxes(title_text='Circularity', range=[0, 0.95], row=row_idx, col=1)

fig.update_xaxes(title_text='Epoch', row=n_variants, col=1)
fig.update_layout(
    title='Circularity by Site — Health Spectrum',
    height=250 * n_variants,
    legend=dict(orientation='h', y=1.02),
)
fig.show()

## Cell 2: MLP Alignment — Does MLP Amplify or Suppress Circular Structure?

**MLP alignment** = `resid_post − attn_out` circularity.

- **Negative**: Adding the MLP output delta reduces circularity — MLP is not aligned with the circular structure in the stream.
- **Positive**: Adding the MLP output delta increases circularity — MLP amplifies the circle.

Healthy models show **3 crossings**: brief positive during initial transient (~epoch 200), returns negative
as attention builds up, then stably positive ~250–350 epochs before grokking.

Metric: **last negative epoch** = when MLP alignment stably commits to positive.

In [ ]:
fig = go.Figure()

crossover_table = []

for v in VARIANTS:
    data = circularity_data[v['label']]
    epochs = data['epochs']
    alignment = data['resid_post'] - data['attn_out']

    # Find the last epoch where alignment is negative (stable crossover point).
    # Healthy models have 3 crossings: brief positive early, then negative as attn
    # builds, then stably positive near grokking. The last negative epoch is the
    # most meaningful marker.
    neg_epochs = epochs[alignment < 0]
    last_neg_ep = int(neg_epochs[-1]) if len(neg_epochs) > 0 else None

    if last_neg_ep and v['grokking_epoch'] > 0:
        lead = v['grokking_epoch'] - last_neg_ep
    else:
        lead = None

    crossover_table.append({
        'variant': v['label'],
        'health': v['health'],
        'grokking_epoch': v['grokking_epoch'],
        'last_negative_epoch': last_neg_ep,
        'lead_before_grokking': lead,
        'n_crossings': int(np.sum(np.diff(np.sign(alignment)) != 0)),
    })

    fig.add_trace(go.Scatter(
        x=epochs, y=alignment,
        mode='lines',
        name=f"{v['label']} ({v['health']})",
        line=dict(color=v['color'], width=2),
        hovertemplate=f"{v['label']}<br>Epoch %{{x}}<br>Alignment: %{{y:+.3f}}<extra></extra>",
    ))

    # Mark last-negative epoch
    if last_neg_ep:
        fig.add_vline(x=last_neg_ep, line=dict(color=v['color'], dash='dot', width=1))

    if v['grokking_epoch'] > 0:
        fig.add_vline(x=v['grokking_epoch'], line=dict(color=v['color'], dash='dash', width=1))

# Zero reference line
fig.add_hline(y=0, line=dict(color='black', width=1, dash='dot'))

fig.update_layout(
    title='MLP Alignment (resid_post − attn_out circularity)<br>'
          'Dashed = grokking epoch  |  Dotted = last negative epoch (stable alignment crossover)',
    xaxis_title='Epoch',
    yaxis_title='MLP Alignment',
    height=500,
)
fig.show()

# Print crossover table
print('MLP Alignment Stable-Crossover Table')
print(f'{"Variant":<25} {"Health":<15} {"Grokking":<12} {"LastNeg":<12} {"Lead":<12} {"Crossings":<10}')
print('-' * 90)
for row in crossover_table:
    gk = str(row['grokking_epoch']) if row['grokking_epoch'] > 0 else 'never'
    ln = str(row['last_negative_epoch']) if row['last_negative_epoch'] else 'none'
    lead = str(row['lead_before_grokking']) if row['lead_before_grokking'] is not None else 'N/A'
    print(f"{row['variant']:<25} {row['health']:<15} {gk:<12} {ln:<12} {lead:<12} {row['n_crossings']:<10}")

## Cell 3: Pre-Grokking Attn Spike

For healthy models, `attn_out` circularity peaks *before* grokking. How far ahead?

Traces `attn_out` for each grokked variant, showing the peak and its timing relative to grokking.

In [ ]:
grokked_variants = [v for v in VARIANTS if v['grokking_epoch'] > 0]

fig = make_subplots(
    rows=len(grokked_variants), cols=1,
    shared_yaxes=False,
    vertical_spacing=0.06,
    subplot_titles=[v['label'] for v in grokked_variants],
)

peak_timing = []

for row_idx, v in enumerate(grokked_variants, 1):
    data = circularity_data[v['label']]
    epochs = data['epochs']
    attn = data['attn_out']
    mlp = data['mlp_out']
    rp = data['resid_post']

    attn_peak_idx = np.argmax(attn)
    attn_peak_ep = int(epochs[attn_peak_idx])
    attn_peak_val = float(attn[attn_peak_idx])
    lead = v['grokking_epoch'] - attn_peak_ep
    peak_timing.append((v['label'], attn_peak_ep, attn_peak_val, v['grokking_epoch'], lead))

    for site, vals, color, label in [
        ('attn_out', attn, '#2ca02c', 'Attn (mid-stream)'),
        ('mlp_out', mlp, '#d62728', 'MLP (output delta)'),
        ('resid_post', rp, '#9467bd', 'Resid Post'),
    ]:
        fig.add_trace(
            go.Scatter(
                x=epochs, y=vals, mode='lines',
                name=label, legendgroup=site,
                showlegend=(row_idx == 1),
                line=dict(color=color, width=2),
            ),
            row=row_idx, col=1,
        )

    # Attn peak marker
    fig.add_vline(x=attn_peak_ep, line=dict(color='#2ca02c', dash='dot', width=2), row=row_idx, col=1)
    # Grokking marker
    fig.add_vline(x=v['grokking_epoch'], line=dict(color='black', dash='dash', width=2), row=row_idx, col=1)
    # Annotation
    fig.add_annotation(
        x=v['grokking_epoch'], y=0.95, xref=f'x{row_idx}', yref=f'y{row_idx}',
        text=f'Grok={v["grokking_epoch"]}',
        showarrow=False, font=dict(size=10),
    )
    fig.add_annotation(
        x=attn_peak_ep, y=0.85, xref=f'x{row_idx}', yref=f'y{row_idx}',
        text=f'Attn peak ({lead} before grok)',
        showarrow=False, font=dict(size=10, color='#2ca02c'),
    )

    fig.update_yaxes(range=[0, 1.0], title_text='Circularity', row=row_idx, col=1)

fig.update_xaxes(title_text='Epoch', row=len(grokked_variants), col=1)
fig.update_layout(
    title='Attn-Out Spike Before Grokking<br>Green dotted = attn peak, Black dashed = grokking',
    height=350 * len(grokked_variants),
    legend=dict(orientation='h', y=1.02),
)
fig.show()

print('\nAttn peak timing relative to grokking:')
print(f'{"Variant":<25} {"Attn peak":<12} {"Peak val":<12} {"Grokking":<12} {"Lead (ep)":<12}')
print('-' * 75)
for label, peak_ep, peak_val, grok_ep, lead in peak_timing:
    print(f'{label:<25} {peak_ep:<12} {peak_val:<12.3f} {grok_ep:<12} {lead:<12}')

## Cell 4: Early Window Δ-Circularity (Epochs 0–500)

Which site shows the largest circularity gain in the initialization-exit window?

Motivated by the observation that `resid_pre` = 0 throughout (embedding contributes no circularity),
so all circularity originates in either attention or MLP processing.

In [ ]:
EARLY_CUTOFF = 500

fig = go.Figure()

variant_labels = [v['label'] for v in VARIANTS]
sites = ['attn_out', 'mlp_out', 'resid_post']
site_colors = ['#2ca02c', '#d62728', '#9467bd']

for site, color in zip(sites, site_colors):
    deltas = []
    for v in VARIANTS:
        data = circularity_data[v['label']]
        epochs = data['epochs']
        circ = data[site]
        early_mask = epochs <= EARLY_CUTOFF
        delta = float(circ[early_mask][-1] - circ[early_mask][0])
        deltas.append(delta)

    fig.add_trace(go.Bar(
        name=SITE_LABELS[site],
        x=variant_labels,
        y=deltas,
        marker_color=color,
    ))

fig.update_layout(
    title=f'Δ-Circularity in Early Window (epochs 0–{EARLY_CUTOFF})',
    xaxis_title='Variant',
    yaxis_title='Δ Circularity',
    barmode='group',
    height=400,
)
fig.show()

# Print table
print(f'Δ-Circularity: epochs 0 → {EARLY_CUTOFF}')
print(f'{"Variant":<25} {"Δ attn_out":<14} {"Δ mlp_out":<14} {"Δ resid_post":<14}')
print('-' * 70)
for v in VARIANTS:
    data = circularity_data[v['label']]
    epochs = data['epochs']
    early_mask = epochs <= EARLY_CUTOFF
    da = float(data['attn_out'][early_mask][-1] - data['attn_out'][early_mask][0])
    dm = float(data['mlp_out'][early_mask][-1] - data['mlp_out'][early_mask][0])
    dr = float(data['resid_post'][early_mask][-1] - data['resid_post'][early_mask][0])
    print(f"{v['label']:<25} {da:<14.3f} {dm:<14.3f} {dr:<14.3f}")

## Cell 5: Phase-Normalized View (Grokked Variants Only)

X-axis = epoch / grokking_epoch. Allows comparison at equivalent developmental stages.
Never-grokked variants are excluded.

Shows whether the attn spike is structurally consistent at a fixed phase fraction.

In [ ]:
grokked = [v for v in VARIANTS if v['grokking_epoch'] > 0]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Attn Out', 'MLP Out', 'Resid Post'],
    shared_yaxes=True,
)

for v in grokked:
    data = circularity_data[v['label']]
    epochs = data['epochs']
    phase = epochs / v['grokking_epoch']

    for col_idx, site in enumerate(['attn_out', 'mlp_out', 'resid_post'], 1):
        fig.add_trace(
            go.Scatter(
                x=phase, y=data[site],
                mode='lines',
                name=v['label'],
                legendgroup=v['label'],
                showlegend=(col_idx == 1),
                line=dict(color=v['color'], width=2),
                hovertemplate=f"{v['label']}<br>Phase %{{x:.2f}}<br>Circ: %{{y:.3f}}<extra></extra>",
            ),
            row=1, col=col_idx,
        )

# Grokking reference line at phase=1
for col_idx in range(1, 4):
    fig.add_vline(x=1.0, line=dict(color='black', dash='dash', width=1), row=1, col=col_idx)

fig.update_xaxes(title_text='Epoch / Grokking Epoch', range=[0, 2.5])
fig.update_yaxes(title_text='Circularity', range=[0, 0.95], col=1)
fig.update_layout(
    title='Phase-Normalized Circularity (grokked variants only)<br>Dashed = grokking (phase=1)',
    height=450,
    legend=dict(orientation='h', y=1.05),
)
fig.show()

## Cell 6: p113 Canon — Pre-Grokking Decoupling Window

p113 shows a 5000-epoch window (epochs 6500–11400) where `attn_out` rises while `mlp_out` falls.
This is the most dramatic MLP-decoupling case in the dataset.

This cell zooms into that window and overlays the grokking epoch.

In [ ]:
p113_data = circularity_data['p113/s999/ds598']
epochs = p113_data['epochs']
grok_ep = 12448

fig = go.Figure()

for site, color, label in [
    ('attn_out', '#2ca02c', 'Attn (mid-stream)'),
    ('mlp_out', '#d62728', 'MLP (output delta)'),
    ('resid_post', '#9467bd', 'Resid Post'),
]:
    fig.add_trace(go.Scatter(
        x=epochs, y=p113_data[site],
        mode='lines', name=label,
        line=dict(color=color, width=2.5),
    ))

# Markers
attn_peak_idx = np.argmax(p113_data['attn_out'])
attn_peak_ep = int(epochs[attn_peak_idx])
attn_peak_val = float(p113_data['attn_out'][attn_peak_idx])

mlp_min_idx = np.argmin(p113_data['mlp_out'][epochs >= 5000][::-1].argmax() + 1)  # rough

fig.add_vline(x=grok_ep, line=dict(color='black', dash='dash', width=2),
              annotation_text=f'Grokking ({grok_ep})', annotation_position='top right')
fig.add_vline(x=attn_peak_ep, line=dict(color='#2ca02c', dash='dot', width=2),
              annotation_text=f'Attn peak ({attn_peak_ep})', annotation_position='top left')

# Shade the decoupling window
fig.add_vrect(
    x0=6500, x1=11400,
    fillcolor='yellow', opacity=0.08,
    layer='below', line_width=0,
    annotation_text='Decoupling window', annotation_position='bottom right',
)

fig.add_hline(y=0, line=dict(color='gray', dash='dot', width=1))

fig.update_layout(
    title='p113/s999/ds598 — Pre-Grokking Decoupling Window<br>Yellow = period where attn rises while MLP falls',
    xaxis_title='Epoch',
    yaxis_title='Circularity',
    yaxis_range=[0, 0.95],
    height=500,
)
fig.show()

# Compute alignment at key moments for p113
print('p113 alignment analysis:')
key_epochs_p113 = [0, 1000, 3000, 6500, 9000, 11400, 12448, 13000, 15000, 20000]
print(f'{"Epoch":>8} {"attn":>8} {"mlp":>8} {"rp":>8} {"align(rp-attn)":>16}')
print('-' * 56)
for e in key_epochs_p113:
    idx = np.searchsorted(epochs, e)
    if idx >= len(epochs):
        continue
    actual_e = int(epochs[idx])
    a = float(p113_data['attn_out'][idx])
    m = float(p113_data['mlp_out'][idx])
    r = float(p113_data['resid_post'][idx])
    print(f'{actual_e:>8} {a:>8.3f} {m:>8.3f} {r:>8.3f} {r-a:>+16.3f}')

## Cell 7: Cross-Variant Survey — MLP Alignment Patterns (All 40 Variants)

Tests whether the 3-crossing / ~250–350 epoch lead pattern generalizes.

Predictions:
- **Healthy grokkers**: n_crossings = 3, lead ~250–350 epochs before grokking
- **Never-grokked / degraded**: n_crossings ≤ 2, alignment stays negative

Expectation: some supercritical-character variants may defy this frame.

In [ ]:
import json

with open('../results/modulo_addition_1layer/variant_registry.json') as f:
    registry = json.load(f)

def variant_artifacts_dir(v):
    return (f"../results/modulo_addition_1layer/"
            f"modulo_addition_1layer_p{v['prime']}_seed{v['model_seed']}_dseed{v['data_seed']}/artifacts")

def compute_alignment_stats(v):
    """Compute MLP alignment stats for one variant."""
    loader = ArtifactLoader(variant_artifacts_dir(v))
    summary = loader.load_summary('repr_geometry')
    epochs = summary['epochs']
    alignment = summary['resid_post_circularity'] - summary['attn_out_circularity']

    neg_mask = alignment < 0
    last_neg_ep = int(epochs[neg_mask][-1]) if neg_mask.any() else None
    n_crossings = int(np.sum(np.diff(np.sign(alignment)) != 0))

    grok = v.get('test_loss_threshold_first_epoch', -1) or -1
    lead = (grok - last_neg_ep) if (last_neg_ep is not None and grok > 0) else None

    # Attn peak timing
    attn = summary['attn_out_circularity']
    attn_peak_ep = int(epochs[np.argmax(attn)])
    attn_peak_val = float(attn.max())
    attn_lead = (grok - attn_peak_ep) if grok > 0 else None

    return {
        'label': f"p{v['prime']}/s{v['model_seed']}/ds{v['data_seed']}",
        'prime': v['prime'],
        'model_seed': v['model_seed'],
        'data_seed': v['data_seed'],
        'health': v['performance_classification'][0],
        'grokking_epoch': grok,
        'n_crossings': n_crossings,
        'last_neg_epoch': last_neg_ep,
        'lead_before_grokking': lead,
        'attn_peak_epoch': attn_peak_ep,
        'attn_peak_val': attn_peak_val,
        'attn_lead_before_grokking': attn_lead,
    }

# Survey all 40 variants
survey = []
for v in registry:
    try:
        survey.append(compute_alignment_stats(v))
    except Exception as e:
        print(f"FAILED: p{v['prime']}/s{v['model_seed']}/ds{v['data_seed']}: {e}")

print(f"Surveyed {len(survey)} variants")

# Group summary
from collections import defaultdict, Counter
by_health = defaultdict(list)
for row in survey:
    by_health[row['health']].append(row)

for health, rows in sorted(by_health.items()):
    crossings = Counter(r['n_crossings'] for r in rows)
    leads = [r['lead_before_grokking'] for r in rows if r['lead_before_grokking'] is not None]
    print(f"\n{health} (n={len(rows)}):")
    print(f"  n_crossings distribution: {dict(sorted(crossings.items()))}")
    if leads:
        print(f"  lead before grokking: min={min(leads)}, median={sorted(leads)[len(leads)//2]}, max={max(leads)}")

In [ ]:
HEALTH_COLORS = {
    'healthy': '#2ca02c',
    'late_grokker': '#ff7f0e',
    'degraded': '#d62728',
    'no_second_descent': '#9467bd',
}

# --- Plot 1: n_crossings vs health (jittered strip) ---
fig = go.Figure()
rng = np.random.default_rng(42)

for health, rows in sorted(by_health.items()):
    x_vals = [r['n_crossings'] + rng.uniform(-0.15, 0.15) for r in rows]
    labels_hover = [r['label'] for r in rows]
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=[health] * len(rows),
        mode='markers',
        name=health,
        marker=dict(color=HEALTH_COLORS.get(health, 'gray'), size=10, opacity=0.75),
        text=labels_hover,
        hovertemplate='%{text}<br>n_crossings: %{x:.0f}<extra></extra>',
    ))

fig.update_layout(
    title='MLP Alignment Zero-Crossings by Health Label',
    xaxis_title='n_crossings',
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    yaxis_title='Health',
    height=350,
)
fig.show()

# --- Plot 2: Lead before grokking vs grokking epoch (grokked only) ---
grokked_rows = [r for r in survey if r['grokking_epoch'] > 0 and r['lead_before_grokking'] is not None]

fig2 = go.Figure()
for health in sorted(HEALTH_COLORS):
    rows_h = [r for r in grokked_rows if r['health'] == health]
    if not rows_h:
        continue
    fig2.add_trace(go.Scatter(
        x=[r['grokking_epoch'] for r in rows_h],
        y=[r['lead_before_grokking'] for r in rows_h],
        mode='markers',
        name=health,
        marker=dict(color=HEALTH_COLORS[health], size=10, opacity=0.8),
        text=[r['label'] for r in rows_h],
        hovertemplate='%{text}<br>Grokking: %{x}<br>Lead: %{y}<extra></extra>',
    ))

fig2.add_hline(y=0, line=dict(color='black', dash='dot', width=1))
fig2.update_layout(
    title='Lead Epochs (MLP Alignment → Grokking) vs Grokking Epoch<br>'
          'Positive = MLP aligns before grokking  |  Negative = MLP still misaligned at grokking',
    xaxis_title='Grokking Epoch',
    yaxis_title='Lead (grokking − last_neg_epoch)',
    height=450,
)
fig2.show()

# --- Full table sorted by health then grokking epoch ---
print(f'\n{"Variant":<25} {"Health":<18} {"Grokking":<10} {"n_cross":<9} {"LastNeg":<10} {"Lead":<8} {"AttnPeak":<10} {"AttnLead":<10}')
print('-' * 105)
for health in ['healthy', 'late_grokker', 'degraded', 'no_second_descent']:
    rows_h = sorted(by_health.get(health, []), key=lambda r: r['grokking_epoch'])
    for r in rows_h:
        gk = str(r['grokking_epoch']) if r['grokking_epoch'] > 0 else 'never'
        ln = str(r['last_neg_epoch']) if r['last_neg_epoch'] else '-'
        ld = f"{r['lead_before_grokking']:+d}" if r['lead_before_grokking'] is not None else 'N/A'
        al = str(r['attn_lead_before_grokking']) if r['attn_lead_before_grokking'] is not None else 'N/A'
        print(f"{r['label']:<25} {health:<18} {gk:<10} {r['n_crossings']:<9} {ln:<10} {ld:<8} {str(r['attn_peak_epoch']):<10} {al:<10}")

## Cell 8: Ring-Plane Organization and the Three Model Characters

The sum `PC1+PC2` fraction of class centroid variance measures how cleanly centroids are organized
onto a 2D plane (ring-like). For a perfect ring: PC1+PC2 → 1.0 (all variance in the ring plane).

**`diff_sum` = attn (PC1+PC2) − mlp (PC1+PC2)**
- Positive: attention's centroids are *more* ring-organized than MLP's
- Negative: MLP's centroids are more ring-organized (or attention is scattered)

Three model characters emerge:
- **Sharp grokker (p109)**: attn ring-sum → 0.977, diff_sum peaks ~+0.49 — near-perfect ring at attn site, MLP actively resists
- **Moderate grokker (p103, n_cross=1)**: attn ring-sum → 0.775, diff_sum peaks ~+0.26 — same opposing structure, smaller amplitude
- **Supercritical (p59)**: diff_sum stays *negative* throughout — MLP's centroids are slightly more ring-organized, but both flat (~0.4–0.5), no crystallization event

In [ ]:
def load_ring_plane_data(variant_dir):
    """Load PC1+PC2 ring-plane variance per site from repr_geometry summary."""
    loader = ArtifactLoader(f'{RESULTS_BASE}/{variant_dir}/artifacts')
    s = loader.load_summary('repr_geometry')
    return {
        'epochs': s['epochs'],
        'attn_ring': s['attn_out_pca_var_pc1'] + s['attn_out_pca_var_pc2'],
        'mlp_ring':  s['mlp_out_pca_var_pc1']  + s['mlp_out_pca_var_pc2'],
        'rp_ring':   s['resid_post_pca_var_pc1'] + s['resid_post_pca_var_pc2'],
        'diff_sum':  (s['attn_out_pca_var_pc1'] + s['attn_out_pca_var_pc2']) -
                     (s['mlp_out_pca_var_pc1']  + s['mlp_out_pca_var_pc2']),
    }

ring_data = {v['label']: load_ring_plane_data(v['dir']) for v in VARIANTS}

# --- Plot 1: diff_sum (attn ring - mlp ring) for all variants ---
fig = go.Figure()

for v in VARIANTS:
    rd = ring_data[v['label']]
    fig.add_trace(go.Scatter(
        x=rd['epochs'], y=rd['diff_sum'],
        mode='lines',
        name=f"{v['label']} ({v['health']})",
        line=dict(color=v['color'], width=2),
        hovertemplate=f"{v['label']}<br>Epoch %{{x}}<br>diff_sum: %{{y:+.3f}}<extra></extra>",
    ))
    if v['grokking_epoch'] > 0:
        fig.add_vline(x=v['grokking_epoch'], line=dict(color=v['color'], dash='dash', width=1))

fig.add_hline(y=0, line=dict(color='black', dash='dot', width=1))
fig.update_layout(
    title='Ring-Plane Divergence: attn(PC1+PC2) − mlp(PC1+PC2)<br>'
          'Positive = attn centroids more ring-organized  |  Dashed = grokking epoch',
    xaxis_title='Epoch',
    yaxis_title='diff_sum',
    height=450,
)
fig.show()

# --- Plot 2: attn_ring and mlp_ring side by side per variant ---
fig2 = make_subplots(
    rows=len(VARIANTS), cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=[f"{v['label']} ({v['health']})" for v in VARIANTS],
)

for row_idx, v in enumerate(VARIANTS, 1):
    rd = ring_data[v['label']]
    epochs = rd['epochs']

    for vals, color, label, dash in [
        (rd['attn_ring'], '#2ca02c', 'Attn PC1+PC2', 'solid'),
        (rd['mlp_ring'],  '#d62728', 'MLP PC1+PC2',  'solid'),
        (rd['rp_ring'],   '#9467bd', 'ResidPost PC1+PC2', 'dot'),
    ]:
        fig2.add_trace(
            go.Scatter(
                x=epochs, y=vals, mode='lines',
                name=label, legendgroup=label,
                showlegend=(row_idx == 1),
                line=dict(color=color, width=2, dash=dash),
                hovertemplate=f"{label}<br>Epoch %{{x}}<br>%{{y:.3f}}<extra></extra>",
            ),
            row=row_idx, col=1,
        )

    if v['grokking_epoch'] > 0:
        fig2.add_vline(x=v['grokking_epoch'], line=dict(color='black', dash='dash', width=1),
                       row=row_idx, col=1)

    # Reference line: perfect ring = 1.0
    fig2.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=row_idx, col=1)
    fig2.update_yaxes(range=[0.2, 1.05], title_text='PC1+PC2', row=row_idx, col=1)

fig2.update_xaxes(title_text='Epoch', row=len(VARIANTS), col=1)
fig2.update_layout(
    title='Ring-Plane Organization Per Site (PC1+PC2 fraction)<br>Gray dotted = perfect ring (1.0)',
    height=230 * len(VARIANTS),
    legend=dict(orientation='h', y=1.02),
)
fig2.show()

# --- Summary table: peak diff_sum and timing ---
print(f'\n{"Variant":<25} {"Health":<15} {"Peak diff_sum":<16} {"Peak epoch":<12} {"Attn ring max":<15} {"MLP ring min":<12}')
print('-' * 95)
for v in VARIANTS:
    rd = ring_data[v['label']]
    ep = rd['epochs']
    ds = rd['diff_sum']
    peak_idx = np.argmax(ds)
    print(f"{v['label']:<25} {v['health']:<15} {ds[peak_idx]:>+14.3f}  "
          f"{int(ep[peak_idx]):<12} {rd['attn_ring'].max():>13.3f}  {rd['mlp_ring'].min():>10.3f}")